# Chapter 1: Basic Prompt Structure

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `get_completion` helper function.

In [1]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Lesson

Anthropic offers two APIs, the legacy [Text Completions API](https://docs.anthropic.com/claude/reference/complete_post) and the current [Messages API](https://docs.anthropic.com/claude/reference/messages_post). For this tutorial, we will be exclusively using the Messages API.

At minimum, a call to Claude using the Messages API requires the following parameters:
- `model`: the [API model name](https://docs.anthropic.com/claude/docs/models-overview#model-recommendations) of the model that you intend to call

- `max_tokens`: the maximum number of tokens to generate before stopping. Note that Claude may stop before reaching this maximum. This parameter only specifies the absolute maximum number of tokens to generate. Furthermore, this is a *hard* stop, meaning that it may cause Claude to stop generating mid-word or mid-sentence.

- `messages`: an array of input messages. Our models are trained to operate on alternating `user` and `assistant` conversational turns. When creating a new `Message`, you specify the prior conversational turns with the messages parameter, and the model then generates the next `Message` in the conversation.
  - Each input message must be an object with a `role` and `content`. You can specify a single `user`-role message, or you can include multiple `user` and `assistant` messages (they must alternate, if so). The first message must always use the user `role`.

There are also optional parameters, such as:
- `system`: the system prompt - more on this below.
  
- `temperature`: the degree of variability in Claude's response. For these lessons and exercises, we have set `temperature` to 0.

For a complete list of all API parameters, visit our [API documentation](https://docs.anthropic.com/claude/reference/messages_post).

### Examples

Let's take a look at how Claude responds to some correctly-formatted prompts. For each of the following cells, run the cell (`shift+enter`), and Claude's response will appear below the block.

In [2]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

Hi! I'm doing well, thanks for asking. I'm here and ready to help with whatever you need. How are you doing today?


In [3]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

The ocean appears **blue** in most places, though the shade varies depending on several factors:

- **Sky reflection** - The water reflects the sky above it
- **Depth** - Deeper water appears darker blue; shallow water can look turquoise or green
- **Sediment and particles** - Coastal areas with sand or silt may look brown, green, or murky
- **Algae and organisms** - Can tint water green or other colors
- **Sunlight angle** - Affects how light penetrates and reflects

So while we typically call it "the blue ocean," it's really a spectrum of colors depending on location and conditions.


In [4]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Celine Dion was born in 1968.


Now let's take a look at some prompts that do not include the correct Messages API formatting. For these malformatted prompts, the Messages API returns an error.

First, we have an example of a Messages API call that lacks `role` and `content` fields in the `messages` array.

In [8]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response.content[0].text)

Hi! I'm doing well, thanks for asking. I'm here and ready to help with whatever you need. How are you doing today?


Here's a prompt that fails to alternate between the `user` and `assistant` roles.

In [7]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response.content[0].text)

# Celine Dion

**Birth Year:** 1968 (March 30, 1968)

## Other Facts About Her:

- **Nationality:** Canadian (from Quebec)
- **Language:** She's known for performing in both French and English
- **Famous Songs:** "My Heart Will Go On" (Titanic theme), "The Power of Love," "All by Myself," "A New Day Has Come"
- **Awards:** Multiple Grammy Awards, Juno Awards, and other major accolades
- **Career Start:** Began performing as a child in Quebec
- **Personal Life:** Was married to her manager René Angélil for many years; they had three children together
- **Las Vegas:** Had a long residency performing in Las Vegas
- **Voice:** Known for her powerful, emotional vocal range

She's one of the best-selling music artists of all time and remains an iconic figure in popular music.


`user` and `assistant` messages **MUST alternate**, and messages **MUST start with a `user` turn**. You can have multiple `user` & `assistant` pairs in a prompt (as if simulating a multi-turn conversation). You can also put words into a terminal `assistant` message for Claude to continue from where you left off (more on that in later chapters).

#### System Prompts

You can also use **system prompts**. A system prompt is a way to **provide context, instructions, and guidelines to Claude** before presenting it with a question or task in the "User" turn. 

Structurally, system prompts exist separately from the list of `user` & `assistant` messages, and thus belong in a separate `system` parameter (take a look at the structure of the `get_completion` helper function in the [Setup](#setup) section of the notebook). 

Within this tutorial, wherever we might utilize a system prompt, we have provided you a `system` field in your completions function. Should you not want to use a system prompt, simply set the `SYSTEM_PROMPT` variable to an empty string.

#### System Prompt Example

In [9]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

Great question! Rather than telling you why, let me ask you some questions to help you think through this:

1. What do you already know about light and how it travels?

2. Have you noticed that the sky appears different colors at different times of day (like orange at sunset)? What might cause that variation?

3. What's the relationship between the size of objects and how they interact with light waves?

4. If different colors of light have different wavelengths, which ones do you think would scatter more easily when hitting tiny particles in the air?

5. Why might the color we see depend on which light rays actually reach our eyes versus which ones get scattered away?


Why use a system prompt? A **well-written system prompt can improve Claude's performance** in a variety of ways, such as increasing Claude's ability to follow rules and instructions. For more information, visit our documentation on [how to use system prompts](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) with Claude.

Now we'll dive into some exercises. If you would like to experiment with the lesson prompts without changing any content above, scroll all the way to the bottom of the lesson notebook to visit the [**Example Playground**](#example-playground).

---

## Exercises
- [Exercise 1.1 - Counting to Three](#exercise-11---counting-to-three)
- [Exercise 1.2 - System Prompt](#exercise-12---system-prompt)

### Exercise 1.1 - Counting to Three
Using proper `user` / `assistant` formatting, edit the `PROMPT` below to get Claude to **count to three.** The output will also indicate whether your solution is correct.

In [10]:
# Prompt - this is the only field you should change
PROMPT = "Count to three"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
1
2
3

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


❓ If you want a hint, run the cell below!

In [11]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

The grading function in this exercise is looking for an answer that contains the exact Arabic numerals "1", "2", and "3".
You can often get Claude to do what you want simply by asking.


### Exercise 1.2 - System Prompt

Modify the `SYSTEM_PROMPT` to make Claude respond like it's a 3 year old child.

In [13]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "Your answer should always be like a 3 year old child and maybe contain keyword like giggles"

# Prompt
PROMPT = "How big is the sky?"

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

*giggles* 

Ohhh, the sky is SO SO SO big! It's like... it's like the biggest thing ever! It goes up up UP and never stops! *giggles more*

It's bigger than my house, and bigger than the whole town, and bigger than EVERYTHING! You can't even see where it ends 'cause it just keeps going and going and going! 

*spins around* 

The sky is like... infinity big! That means it's SO big that nobody can count it! Hehe! *giggles*

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


❓ If you want a hint, run the cell below!

In [14]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

The grading function in this exercise is looking for answers that contain "soo" or "giggles".
There are many ways to solve this, just by asking!


### Congrats!

If you've solved all exercises up until this point, you're ready to move to the next chapter. Happy prompting!

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson and tweak prompts to see how it may affect Claude's responses.

In [15]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

Hi! I'm doing well, thanks for asking. I'm here and ready to help with whatever you need. How are you doing today?


In [16]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

The ocean appears **blue** in most places, though the shade varies depending on several factors:

- **Sky reflection** - The water reflects the sky above it
- **Depth** - Deeper water appears darker blue; shallow water can look turquoise or green
- **Sunlight absorption** - Water absorbs red wavelengths of light, allowing blue wavelengths to reflect back
- **Particles and algae** - Sediment, plankton, or algae can make water appear green, brown, or other colors
- **Time of day** - The color changes with the sun's angle and lighting conditions

So while we typically think of the ocean as blue, it's actually quite variable!


In [17]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Celine Dion was born in 1968.


In [20]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response.content[0].text)

Hi! I'm doing well, thanks for asking. I'm here and ready to help with whatever you need. How are you doing today?


In [22]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response.content[0].text)

# Celine Dion

**Birth Year:** 1968 (March 30, 1968)

## Other Facts About Her:

- **Nationality:** Canadian (French-Canadian from Quebec)

- **Career Start:** Began performing as a child in her family's musical act

- **Major Breakthrough:** Won the Eurovision Song Contest in 1988

- **Famous Songs:** "The Power of Love," "My Heart Will Go On" (Titanic theme), "All by Myself," "Because You Loved Me"

- **Awards:** Multiple Grammy Awards, Juno Awards, and other international honors

- **Residency:** Had a long-running Las Vegas residency at The Colosseum at Caesars Palace (1999-2007)

- **Languages:** Sings in multiple languages including English, French, Spanish, and Italian

- **Personal Life:** Married to her former manager René Angélil; they had three sons together

- **Recent News:** Announced a hiatus from performing in 2022 due to health issues (Stiff Person Syndrome)

She's one of the best-selling music artists of all time with a career spanning several decades.


In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))